In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/shardulsenpai/phase2-math500-3b-results/phase2_math500_3B_results.json


In [ ]:
import torch
import json
import re
import gc
import os
import numpy as np
import matplotlib.pyplot as plt
from transformers import AutoModelForCausalLM, AutoTokenizer
from scipy.stats import kendalltau

# --- KAGGLE CONFIGURATION ---
# IMPORTANT: Adjust 'gsm8k-subset' if you named your uploaded dataset differently
DATA_PATH = "/kaggle/input/datasets/shardulsenpai/gsm8k-subset/gsm8k_subset.json"
OUTPUT_PATH = "/kaggle/working/pilot_results.json"
MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"
DTYPE = torch.bfloat16

def load_environment():
    print(f"Loading {MODEL_ID} on Kaggle GPUs...")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        dtype=DTYPE,
        device_map="auto",           # Utilizes Kaggle GPUs
        attn_implementation="eager"  # Required for raw attention extraction
    )
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    return model, tokenizer

def calculate_token_entropies(logits):
    probs = torch.softmax(logits, dim=-1)
    epsilon = 1e-9
    return -torch.sum(probs * torch.log2(probs + epsilon), dim=-1)

def align_steps_to_tokens(generated_text, full_token_ids, tokenizer, prompt_len):
    encoding = tokenizer(generated_text, add_special_tokens=False, return_offsets_mapping=True)
    offsets = encoding.offset_mapping
    
    generated_ids = full_token_ids[prompt_len:]
    if len(generated_ids) - len(offsets) == 1:
        generated_ids = generated_ids[:-1]
    elif len(offsets) != len(generated_ids):
        return None, None
        
    step_pattern = re.compile(r"(?m)^(Step \d+:.*?)(?=^Step \d+:|^Final Answer:|\Z)", re.DOTALL)
    step_spans = []
    
    for match in step_pattern.finditer(generated_text):
        char_start, char_end = match.span(1)
        start_token_idx = None
        end_token_idx = None
        
        for idx, (tok_char_start, tok_char_end) in enumerate(offsets):
            if start_token_idx is None and tok_char_start <= char_start < tok_char_end:
                start_token_idx = idx
            if tok_char_end <= char_end:
                end_token_idx = idx
                
        if start_token_idx is not None and end_token_idx is not None:
            step_spans.append((start_token_idx + prompt_len, end_token_idx + 1 + prompt_len))
            
    return step_spans, generated_ids

def plot_trajectories(out_entropies, attn_entropies, attn_norm, problem_id):
    steps = list(range(1, len(out_entropies) + 1))
    fig, ax1 = plt.subplots(figsize=(10, 6))
    
    color1 = 'tab:red'
    ax1.set_xlabel('Reasoning Step')
    ax1.set_ylabel('Output Entropy', color=color1)
    ax1.plot(steps, out_entropies, marker='o', color=color1, label='Output')
    ax1.tick_params(axis='y', labelcolor=color1)
    
    ax2 = ax1.twinx()  
    color2 = 'tab:blue'
    color3 = 'tab:green'
    ax2.set_ylabel('Attention Entropy', color='black')
    ax2.plot(steps, attn_entropies, marker='s', linestyle='--', color=color2, label='Attn (Raw)')
    ax2.plot(steps, attn_norm, marker='^', color=color3, label='Attn (Norm)')
    ax2.tick_params(axis='y', labelcolor='black')
    
    lines_1, labels_1 = ax1.get_legend_handles_labels()
    lines_2, labels_2 = ax2.get_legend_handles_labels()
    ax1.legend(lines_1 + lines_2, labels_1 + labels_2, loc='upper right')
    
    plt.title(f'Entropy Trajectories (GSM8K ID: {problem_id})')
    fig.tight_layout()
    plt.grid(True, alpha=0.3)
    
    plot_path = f"/kaggle/working/trajectory_pilot_{problem_id}.png"
    plt.savefig(plot_path, dpi=300)
    print(f"\n[PLOT SAVED] Generated: {plot_path}")
    plt.close() 

def process_pilot(model, tokenizer, dataset):
    if not os.path.exists(OUTPUT_PATH):
        with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
            json.dump([], f)

    for i, problem in enumerate(dataset):
        print(f"\n{'='*50}\n--- Processing Problem {problem['id']} ({i+1}/{len(dataset)}) ---\n{'='*50}")
        
        try:
            with open(OUTPUT_PATH, "r", encoding="utf-8") as f:
                existing_records = json.load(f)
            if any(record["id"] == problem["id"] for record in existing_records):
                print(f"Problem {problem['id']} already found in JSON. Skipping to next.")
                continue
        except (FileNotFoundError, json.JSONDecodeError):
            existing_records = []

        system_prompt = (
            "You are a strict mathematical reasoning engine. You must solve the problem step-by-step. "
            "Every single logical step MUST begin on a new line with 'Step N:'. "
            "When you finish reasoning, you MUST end your response with the exact format 'Final Answer: [number]' and nothing else."
        )
        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": problem["question"]}
        ]
        
        text_prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = tokenizer([text_prompt], return_tensors="pt").to(model.device)
        prompt_len = inputs.input_ids.shape[1]
        
        # --- PASS 1: GENERATION ---
        with torch.no_grad():
            outputs_gen = model.generate(
                **inputs,
                max_new_tokens=1024,
                do_sample=False,
                return_dict_in_generate=True,
                output_scores=True
            )
            
        full_seq = outputs_gen.sequences[0]
        gen_ids = full_seq[prompt_len:]
        logits = torch.cat(outputs_gen.scores, dim=0)
        
        generated_text = tokenizer.decode(gen_ids, skip_special_tokens=True)
        step_spans, clean_gen_ids = align_steps_to_tokens(generated_text, full_seq, tokenizer, prompt_len)
        
        if not step_spans:
            print("[DEBUG] Alignment mapping failed. Skipping problem.")
            continue
            
        reencoded_ids = tokenizer(generated_text, add_special_tokens=False).input_ids
        if reencoded_ids != clean_gen_ids.tolist():
            print(f"[DEBUG] BPE Mismatch! Re-encoded: {len(reencoded_ids)}, Clean original: {len(clean_gen_ids)}. Skipping problem.")
            continue
            
        if len(logits) - len(clean_gen_ids) == 1:
            logits = logits[:-1]
            
        out_token_entropies = calculate_token_entropies(logits)
        
        del outputs_gen
        torch.cuda.empty_cache()
        gc.collect()

        # --- PASS 2: ATTENTION EXTRACTION ---
        print("Extracting full attention matrices...")
        pass2_inputs = full_seq.unsqueeze(0).to(model.device)
        seq_len = pass2_inputs.shape[1]
        
        with torch.no_grad():
            outputs_extract = model(
                pass2_inputs,
                output_attentions=True,
                use_cache=False 
            )
            
        attentions = outputs_extract.attentions
        
        if i == 0:
            print(f"\n[CHECK 1 & 2] Tensor Shapes")
            print(f"Total Layers: {len(attentions)}")
            print(f"Sequence Length: {seq_len}")
            
            allocated_vram = torch.cuda.memory_allocated(0) / (1024**3)
            print(f"\n[CHECK 4] VRAM Allocated (GPU 0): {allocated_vram:.2f} GB")
            
            print("\n[CHECK 3] Attention Row Sums (Sample layers)")
            for l_idx in [0, 14, 27]:
                row_sums = attentions[l_idx][0].sum(dim=-1)
                print(f"Layer {l_idx:02d} | Min: {row_sums.min().item():.4f}, Max: {row_sums.max().item():.4f}")
            
        epsilon = 1e-9
        layer_entropies = []
        
        n_attended_tokens = torch.arange(1, seq_len + 1, device=model.device, dtype=DTYPE)
        max_entropy_bounds = torch.log2(n_attended_tokens)
        max_entropy_bounds = torch.clamp(max_entropy_bounds, min=1e-9) 
        
        for l in range(len(attentions)):
            attn_probs = attentions[l][0] 
            head_entropies = -torch.sum(attn_probs * torch.log2(attn_probs + epsilon), dim=-1) 
            mean_head_entropy = head_entropies.mean(dim=0)
            layer_entropies.append(mean_head_entropy)
            
        global_attn_entropy = torch.stack(layer_entropies).mean(dim=0)
        global_attn_entropy_norm = global_attn_entropy / max_entropy_bounds
        
        # --- STEP AGGREGATION ---
        step_out_entropies = []
        step_attn_global = []
        step_attn_global_norm = []
        step_attn_std = []
        step_token_lengths = []
        step_layer_dict = {f"layer_{l}": [] for l in range(len(attentions))}
        
        for start_idx, end_idx in step_spans:
            step_length = end_idx - start_idx
            step_token_lengths.append(step_length)
            
            out_span = out_token_entropies[start_idx - prompt_len : end_idx - prompt_len]
            step_out_entropies.append(out_span.mean().item())
            
            attn_span = global_attn_entropy[start_idx:end_idx]
            step_attn_global.append(attn_span.mean().item())
            step_attn_std.append(attn_span.std().item() if len(attn_span) > 1 else 0.0)
            
            attn_span_norm = global_attn_entropy_norm[start_idx:end_idx]
            step_attn_global_norm.append(attn_span_norm.mean().item())
            
            for l in range(len(attentions)):
                layer_span = layer_entropies[l][start_idx:end_idx]
                step_layer_dict[f"layer_{l}"].append(layer_span.mean().item())
                
        if i == 0:
            plot_trajectories(step_out_entropies, step_attn_global, step_attn_global_norm, problem["id"])
            
        out_tau, _ = kendalltau(list(range(len(step_out_entropies))), step_out_entropies)
        attn_tau, _ = kendalltau(list(range(len(step_attn_global_norm))), step_attn_global_norm)
        
        # --- PROGRESSIVE SAVE FILE UPDATE ---
        new_record = {
            "id": problem["id"],
            "total_seq_len": seq_len,
            "step_token_lengths": step_token_lengths,
            "out_tau": out_tau if not np.isnan(out_tau) else 0.0,
            "attn_tau": attn_tau if not np.isnan(attn_tau) else 0.0,
            "step_out_mean": step_out_entropies,
            "step_attn_global": step_attn_global,
            "step_attn_global_norm": step_attn_global_norm,
            "step_attn_std": step_attn_std,
            "layer_wise_attn": step_layer_dict
        }
        
        try:
            with open(OUTPUT_PATH, "r", encoding="utf-8") as f:
                current_records = json.load(f)
        except (FileNotFoundError, json.JSONDecodeError):
            current_records = []
            
        current_records.append(new_record)
        
        with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
            json.dump(current_records, f, indent=4)
            
        print(f"[PROGRESS LOCKED] Problem {problem['id']} successfully appended to /kaggle/working/pilot_results.json")
        
        del outputs_extract
        del attentions
        torch.cuda.empty_cache()
        gc.collect()

    print("\nExtraction script completely finished execution loop.")

if __name__ == "__main__":
    from datasets import load_dataset
    
    print("Downloading MATH-500 directly from HuggingFace...")
    # This downloads the standard 500-question test set used by the community
    hf_dataset = load_dataset("HuggingFaceH4/MATH-500", split="test")
    
    # Convert it into the exact format your pipeline expects
    dataset = []
    for i, item in enumerate(hf_dataset):
        dataset.append({
            "id": f"math500_{i}",
            "question": item["problem"],
            "answer": item["solution"]
        })
        
    print(f"Successfully loaded {len(dataset)} MATH-500 problems.")
    
    # Run the pipeline!
    model, tokenizer = load_environment()
    process_pilot(model, tokenizer, dataset)

README.md:   0%|          | 0.00/412 [00:00<?, ?B/s]

test.jsonl: 0.00B [00:00, ?B/s]

Generating test split:   0%|          | 0/500 [00:00<?, ? examples/s]

Successfully loaded 500 MATH-500 problems.
Loading Qwen/Qwen2.5-3B-Instruct on Kaggle GPUs...


config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

...

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.



--- Processing Problem math500_0 (1/500) ---
Extracting full attention matrices...

[CHECK 1 & 2] Tensor Shapes
Total Layers: 36
Sequence Length: 365

[CHECK 4] VRAM Allocated (GPU 0): 3.27 GB

[CHECK 3] Attention Row Sums (Sample layers)
Layer 00 | Min: 0.9961, Max: 1.0000
Layer 14 | Min: 0.9961, Max: 1.0000
Layer 27 | Min: 0.9961, Max: 1.0000

[PLOT SAVED] Generated: /kaggle/working/trajectory_pilot_math500_0.png
[PROGRESS LOCKED] Problem math500_0 successfully appended to /kaggle/working/pilot_results.json

--- Processing Problem math500_1 (2/500) ---
Extracting full attention matrices...
[PROGRESS LOCKED] Problem math500_1 successfully appended to /kaggle/working/pilot_results.json

--- Processing Problem math500_2 (3/500) ---
Extracting full attention matrices...
[PROGRESS LOCKED] Problem math500_2 successfully appended to /kaggle/working/pilot_results.json

--- Processing Problem math500_3 (4/500) ---
Extracting full attention matrices...
[PROGRESS LOCKED] Problem math500_3 succ

In [3]:
import json
import re
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from tqdm.notebook import tqdm # Notebook-friendly progress bar

# --- CONFIGURATION ---
INPUT_JSON = "/kaggle/input/datasets/shardulsenpai/phase2-math500-3b-results/phase2_math500_3B_results.json"
OUTPUT_JSON = "/kaggle/working/phase2_math500_3B_graded.json"
MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"

print("1. Loading your 254 extracted features...")
with open(INPUT_JSON, "r", encoding="utf-8") as f:
    extracted_data = json.load(f)

print("2. Downloading MATH-500 questions from HuggingFace...")
hf_dataset = load_dataset("HuggingFaceH4/MATH-500", split="test")
questions_dict = {f"math500_{i}": item for i, item in enumerate(hf_dataset)}

print(f"3. Loading {MODEL_ID} for fast text generation...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, 
    torch_dtype=torch.bfloat16, 
    device_map="auto"
)

system_prompt = (
    "You are a strict mathematical reasoning engine. You must solve the problem step-by-step. "
    "Every single logical step MUST begin on a new line with 'Step N:'. "
    "When you finish reasoning, you MUST end your response with the exact format 'Final Answer: [number]' and nothing else."
)

graded_data = []

for record in tqdm(extracted_data, desc="Grading Trajectories"):
    q_id = record["id"]
    question_text = questions_dict[q_id]["problem"]
    ground_truth = questions_dict[q_id]["solution"]
    
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": question_text}
    ]
    text_prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer([text_prompt], return_tensors="pt").to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs, 
            max_new_tokens=1024, 
            do_sample=False  # Crucial: Ensures exact match with Kaggle run
        )
        
    gen_ids = outputs[0][inputs.input_ids.shape[1]:]
    generated_text = tokenizer.decode(gen_ids, skip_special_tokens=True)
    
    # Extract final answer
    match = re.search(r"(?i)Final Answer:\s*(.+)", generated_text)
    model_ans = match.group(1).strip() if match else ""
    
    # Basic inclusion grading
    clean_model_ans = model_ans.replace(" ", "").lower()
    clean_truth = ground_truth.replace(" ", "").lower()
    is_correct = 1 if clean_model_ans and (clean_model_ans in clean_truth or clean_truth in clean_model_ans) else 0
    
    # Stitch labels into the existing record
    record["model_generated_answer"] = model_ans
    record["ground_truth_answer"] = ground_truth
    record["is_correct"] = is_correct
    graded_data.append(record)

# Save the final file
with open(OUTPUT_JSON, "w", encoding="utf-8") as f:
    json.dump(graded_data, f, indent=4)
        
print(f"\nDONE! Fully graded JSON saved to {OUTPUT_JSON}")

1. Loading your 254 extracted features...
2. Downloading MATH-500 questions from HuggingFace...


README.md:   0%|          | 0.00/412 [00:00<?, ?B/s]

test.jsonl: 0.00B [00:00, ?B/s]

Generating test split:   0%|          | 0/500 [00:00<?, ? examples/s]

3. Loading Qwen/Qwen2.5-3B-Instruct for fast text generation...


...

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Grading Trajectories:   0%|          | 0/254 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.



DONE! Fully graded JSON saved to /kaggle/working/phase2_math500_3B_graded.json
